# 改訂単体法

In [1]:
import numpy as np

## 最大係数規則

In [12]:
class RevisedSimplex:
    def __init__(self, A, b, c):
        self.A = A
        self.b = b
        self.c = c

        self.m, self.n = A.shape

        self.B_idx = list(range(self.n - self.m, self.n))
        self.N_idx = list(range(self.n - self.m))

    def solve(self):
        iteration = 0
        while True:
            # --- Step 1 ---
            B = self.A[:, self.B_idx] #基底行列
            N = self.A[:, self.N_idx] #非基底行列

            B_inv = np.linalg.inv(B) #基底行列の逆行列
            x_B = B_inv @ self.b  #初期の実行可能基底解
            b_bar = x_B
            
            if np.any(b_bar < 0):   # 実行可能かどうかのチェック
                print("初期実行可能解を見つけられません")
                return None, None, iteration
            # --- Step 2 ---
            c_B = self.c[self.B_idx]
            c_N = self.c[self.N_idx]

            c_bar = c_N - N.T @ B_inv.T @ c_B   # 被約費用の計算

            # --- Step 3 ---
            if np.all(c_bar <= 0): #c_barが全て非正であれば最適解
                x = np.zeros(self.n)
                x[self.B_idx] = b_bar
                return x, self.c @ x, iteration #最適解と最適値と反復回数を返して終了

            # c_barの中で最大のものを選び，対応する非基底変数を基底に入れる（最大係数規則）
            k = np.argmax(c_bar)
            entering = self.N_idx[k]

            # --- Step 4 ---
            a_k = self.A[:, entering]
            a_bar = B_inv @ a_k #N_barの非基底変数に対応する列ベクトル

            if np.all(a_bar <= 0):
               print("非有界です。")
               return None, np.inf, iteration #非有界なので終了

            # --- Step 5 ---
            theta = []
            for i in range(self.m):
                if a_bar[i] > 0:
                    theta.append(b_bar[i] / a_bar[i])
                else:
                    theta.append(np.inf)

            i_star = np.argmin(theta)
            leaving = self.B_idx[i_star]

            # 基底更新
            self.B_idx[i_star] = entering
            self.N_idx[k] = leaving
            iteration += 1

            if iteration > 10000: 
                print("巡回が発生しました。")
                return None, None, iteration

### 例1
$$
\begin{align*}
\text{最大化}\quad&z = x_1 + 2x_2 \\
\text{条件} \quad& x_1 + x_2 \leq 6 \\
\quad& x_1 + 3x_2 \leq 12 \\
\quad& 2x_1 + x_2 \leq 10\\
\quad& x_1, x_2 \geq 0
\end{align*}
$$

In [4]:
A = np.array([[1, 1, 1, 0, 0],
              [1, 3, 0, 1, 0],
              [2, 1, 0, 0, 1]], dtype=float)
b = np.array([6, 12, 10], dtype=float)
c = np.array([1, 2, 0, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

### 例2　非有界のとき
$$
\begin{align*}
\text{最大化}\quad&z = 2x_1 + x_2 \\
\text{条件} \quad& x_1 -2x_2 \leq 4 \\
\quad& -x_1 + x_2 \leq 2 \\
\quad& x_1, x_2 \geq 0
\end{align*}
$$

In [10]:
A = np.array([
    [1, -2, 1, 0],
    [-1, 1, 0, 1]], dtype=float)
b = np.array([4, 2], dtype=float)
c = np.array([2, 1, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

### 例3　巡回に陥るとき
$$
\begin{align*}
\text{最大化}\quad&z = 10x_1 -57x_2 -9x_3 -24x_4 \\
\text{条件} \quad& 0.5x_1 -5.5x_2 -2.5x_3 +9x_4 \leq 0 \\
\quad& 0.5x_1 -1.5x_2 -0.5x_3 + x_4 \leq 0 \\
\quad& x_1 \leq 1 \\
\quad& x_1, x_2, x_3, x_4 \geq 0
\end{align*}
$$


In [13]:
A = np.array([[0.5, -5.5, -2.5, 9, 1, 0, 0], 
              [0.5, -1.5, -0.5, 1, 0, 1, 0],
              [1, 0, 0, 0, 0, 0, 1]], dtype=float)
b = np.array([0, 0, 1], dtype=float)
c = np.array([10, -57, -9, -24, 0, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

### 教科書p.72の2.4をpythonに解いてもらいましょう

(1)
$$
\begin{aligned}
\text{最大化}\quad &4x_1+8x_2+10x_3\\
\text{条件} \quad &x_1+x_2+x_3\leq 20,\\
&3x_1+4x_2+6x_3\leq 100,\\
&4x_1+5x_2+3x_3\leq 100,\\
&x_1,x_2,x_3\geq 0.
\end{aligned}
$$

In [11]:
A= np.array([[1, 1, 1, 1, 0, 0],
                [3, 4, 6, 0, 1, 0],
                [4, 5, 3, 0, 0, 1]], dtype=float)
b = np.array([20, 100, 100], dtype=float)
c = np.array([4, 8, 10, 0, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

(2)　　

$$
\begin{aligned}
\text{最大化}\quad &x_1+3x_2-x_3\\
\text{条件} \quad &2x_1+2x_2-x_3\leq 10,\\
&3x_1-2x_2+x_3\leq 10,\\
&x_1-3x_2+x_3\leq 10,\\
&x_1,x_2,x_3\geq 0.
\end{aligned}
$$

In [13]:
A = np.array([[2, 2, -1, 1, 0, 0],
                [3, -2, 1, 0, 1, 0],
                [1, -3, 1, 0, 0, 1]], dtype=float)
b = np.array([10, 10, 10], dtype=float)
c = np.array([1, 3, -1, 0, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

(3)
$$
\begin{aligned}
\text{最大化}\quad &10x_1+x_2\\
\text{条件} \quad &x_1 \leq 1,\\
&20x_1+x_2 \leq 100,\\
&x_1,x_2\geq 0.
\end{aligned}
$$

In [15]:
A= np.array([[1, 0, 1, 0],
             [20, 1, 0, 1]], dtype=float)
b = np.array([1, 100], dtype=float)
c = np.array([10, 1, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

## 最小添字規則

In [15]:
class RevisedSimplex:
    def __init__(self, A, b, c):
        self.A = A
        self.b = b
        self.c = c

        self.m, self.n = A.shape

        self.B_idx = list(range(self.n - self.m, self.n))
        self.N_idx = list(range(self.n - self.m))

    def solve(self):
        iteration = 0
        while True:
        # --- Step 1 ---
         B = self.A[:, self.B_idx]
         N = self.A[:, self.N_idx]

         B_inv = np.linalg.inv(B)
         b_bar = B_inv @ self.b
         # 実行可能かどうかのチェック
         if np.any(b_bar < 0):
                print("初期実行可能解を見つけられません")
                return None, None, iteration
         

        # --- Step 2 ---
         c_B = self.c[self.B_idx]
         c_N = self.c[self.N_idx]

         c_bar = c_N - N.T @ B_inv.T @ c_B

        # --- Step 3 ---
         candidates = [j for j in range(len(c_bar)) if c_bar[j] > 0]

         if len(candidates) == 0:
            x = np.zeros(self.n)
            x[self.B_idx] = b_bar
            return x, self.c @ x, iteration

        # 最小添字規則
         k = min(candidates)
         entering = self.N_idx[k] #次に基底変数になる非基底変数の添字

        # --- Step 4 ---
         a_k = self.A[:, entering]
         a_bar = B_inv @ a_k

         if np.all(a_bar <= 0):
            return None, None, iteration  # 非有界

        # --- Step 5 ---
         theta = []
         for i in range(self.m):
            if a_bar[i] > 0:
                theta.append(b_bar[i] / a_bar[i])
            else:
                theta.append(np.inf)

         theta_min = min(theta)

         candidates = [i for i in range(self.m) if abs(theta[i] - theta_min) <= 0]
         i_star = min(candidates, key=lambda i: self.B_idx[i])

         leaving = self.B_idx[i_star] #次に非基底変数になる基底変数の添字

        # 基底更新
         self.B_idx[i_star] = entering
         self.N_idx[k] = leaving
         iteration += 1

         if iteration > 10000:
            print("巡回が発生しました。")
            return None, None, iteration
        

### 最大係数規則では巡回に陥った例3を最小添字規則で解く
$$
\begin{align*}
\text{最大化}\quad&z = 10x_1 -57x_2 -9x_3 -24x_4 \\
\text{条件} \quad& 0.5x_1 -5.5x_2 -2.5x_3 +9x_4 \leq 0 \\
\quad& 0.5x_1 -1.5x_2 -0.5x_3 + x_4 \leq 0 \\
\quad& x_1 \leq 1 \\
\quad& x_1, x_2, x_3, x_4 \geq 0
\end{align*}
$$

In [ ]:
A = np.array([[0.5, -5.5, -2.5, 9, 1, 0, 0], 
              [0.5, -1.5, -0.5, 1, 0, 1, 0],
              [1, 0, 0, 0, 0, 0, 1]], dtype=float)
b = np.array([0, 0, 1], dtype=float)
c = np.array([10, -57, -9, -24, 0, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

### クレー・ミンティの問題

In [18]:
def klee_minty(n):
    A = np.zeros((n, 2*n), dtype=float)
    b = np.zeros(n, dtype=float)
    c = np.zeros(2*n, dtype=float)

    for i in range(n):

        for j in range(i):

            if j == i:
                A[i, j] = 1
            else:
                A[i, j] =2*  10**(i-j)

        A[i, i] = 1

        A[i, n+i] = 1

        b[i] = 100**i

    for i in range(n):
        c[i] = 10**(n-i-1)

    return A, b, c

In [ ]:
for k in range(1, 11):
    A, b, c = klee_minty(k)
    solver = RevisedSimplex(A, b, c)
    x_opt, val, iteration = solver.solve()
    print(f"n = {k}:")
    print("iteration =", iteration)

# 2段階単体法

### 例4 "簡単な"初期実行可能解がないとき($x_1=x_2=0$としても実行可能解が得られないとき)
$$
\begin{align*}
\text{最大化}\quad&z = x_1 + 2x_2 \\
\text{条件} \quad& x_1 + x_2 \leq 6 \\
\quad& x_1 + 3x_2 \leq 12 \\
\quad& -3x_1 - 2x_2 \leq -6\\
\quad& x_1, x_2 \geq 0
\end{align*}
$$

In [ ]:
A= np.array([[1, 1, 1, 0, 0],
                [1, 3, 0, 1, 0],
                [-3, -2, 0, 0, 1]], dtype=float)
b = np.array([6, 12, -6], dtype=float)
c = np.array([1, 2, 0, 0, 0], dtype=float)

In [ ]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

## Phase 1: 初期実行可能解の生成

In [68]:
class AuxiliaryProblem: #補助問題

    def __init__(self, A, b):

        self.A = A.astype(float)
        self.b = b.astype(float)

        self.m, self.n = A.shape

    def find_feasible_basis(self):

        A_aux = np.hstack([-np.ones((self.m, 1)),self.A]) #人工変数を導入
        c_aux = np.zeros(self.n + 1)
        c_aux[0] = -1

        B_idx = list(range(self.n - self.m + 1,self.n + 1))
        r = np.argmin(self.b)
        B_idx[r] = 0
        N_idx = [j for j in range(self.n + 1) if j not in B_idx]
        iteration = 0

        while True:
            B = A_aux[:, B_idx]
            N = A_aux[:, N_idx]

            B_inv = np.linalg.inv(B)
            x_B = B_inv @ self.b
            c_B = c_aux[B_idx]
            c_N = c_aux[N_idx]
            c_bar = c_N - N.T @ B_inv.T @ c_B

            if np.all(c_bar <= 0):

                x = np.zeros(self.n + 1)

                x[B_idx] = x_B
                break

            k = np.argmax(c_bar) #最大係数規則

            entering = N_idx[k]
            a_k = A_aux[:, entering]

            a_bar = B_inv @ a_k

            theta = []

            for i in range(self.m):

                if a_bar[i] > 0:
                    theta.append(x_B[i] / a_bar[i])

                else:
                    theta.append(np.inf)

            i_star = np.argmin(theta)
            leaving = B_idx[i_star]
            B_idx[i_star] = entering
            N_idx[k] = leaving
            iteration += 1
        if x[0] > 1e-8:
            raise ValueError("実行可能な基底解が見つかりませんでした。")
            

        feasible_B_idx = []

        for idx in B_idx:

            if idx != 0:

                feasible_B_idx.append(idx - 1)

        feasible_N_idx = []

        for idx in N_idx:
            
            if idx != 0:
                feasible_N_idx.append(idx - 1)

        return feasible_B_idx, feasible_N_idx

## Phase 2: 初期解を改訂単体法に渡す

In [ ]:
A = np.array([
    [1, 1, 1, 0, 0],
    [1, 3, 0, 1, 0],
    [-3, -2, 0, 0, 1]], dtype=float)
b = np.array([6, 12, -6], dtype=float)
c = np.array([1, 2, 0, 0, 0], dtype=float)


# 段階1
phase1 = AuxiliaryProblem(A, b)
B_idx, N_idx = phase1.find_feasible_basis()

# 段階2（通常の改訂単体法へ渡す）

solver = RevisedSimplex(A, b, c)
solver.B_idx,solver.N_idx = B_idx, N_idx

print("初期基底変数の添字:", [i+1 for i in B_idx])
print("初期非基底変数の添字:", [i+1 for i in N_idx])

x_opt, val_opt, iteration = solver.solve()

print("\n最適解:")
print(x_opt)

print("\n最適値:")
print(val_opt)

### 2.5をpythonに解いてもらいましょう

(1)　　
$$
\begin{aligned}
\text{最小化}\quad &2x_1+3x_2+x_3\\
\text{条件} \quad& x_1 + 4x_2 + 2x_3 \geq 8,\\
& 3x_1 + 2x_2 \geq 6\\
& x_1, x_2, x_3 \geq 0.
\end{aligned}
$$

In [ ]:
#最小化，不等号の向きに注意して書き換える
A = np.array([[-1, -4, -2, 1, 0],
            [-3, -2, 0, 0, 1]], dtype=float)
b = np.array([-8, -6], dtype=float)
c = np.array([-2, -3, -1, 0, 0], dtype=float)

# 段階1
phase1 = AuxiliaryProblem(A, b)
B_idx, N_idx = phase1.find_feasible_basis()

# 段階2（通常の改訂単体法へ渡す）

solver = RevisedSimplex(A, b, c)
solver.B_idx,solver.N_idx = B_idx, N_idx

print("初期基底変数の添字:", [i+1 for i in B_idx])
print("初期非基底変数の添字:", [i+1 for i in N_idx])

x_opt, val_opt, iteration = solver.solve()

print("\n最適解:")
print(x_opt)

print("\n最適値:")
print(-val_opt) #目的関数の符号を反転させているため、最適値も反転させる必要がある

(2)　　
$$
\begin{aligned}
\text{最大化}\quad &x_1-x_2+x_3\\
\text{条件} \quad& 2x_1 - x_2 + 2x_3 \leq 4,\\
& 2x_1 -3x_2 + x_3 \leq -5,\\
& -x_1 + x_2 -2x_3 \leq -1,\\
& x_1, x_2, x_3 \geq 0.
\end{aligned}
$$

In [ ]:
A = np.array([[2, -1, 2, 1, 0, 0],
            [2, -3, 1, 0, 1, 0],
            [-1, 1, -2, 0, 0, 1]], dtype=float)
b = np.array([4, -5, -1], dtype=float)
c = np.array([1, -1, 1, 0, 0, 0], dtype=float)

# 段階1
phase1 = AuxiliaryProblem(A, b)
B_idx, N_idx = phase1.find_feasible_basis()

# 段階2（通常の改訂単体法へ渡す）

solver = RevisedSimplex(A, b, c)
solver.B_idx,solver.N_idx = B_idx, N_idx

print("初期基底変数の添字:", [i+1 for i in B_idx])
print("初期非基底変数の添字:", [i+1 for i in N_idx])

x_opt, val_opt, iteration = solver.solve()

print("\n最適解:")
print(x_opt)

print("\n最適値:")
print(val_opt) 

(3)　　
$$
\begin{aligned}
\text{最大化}\quad &3x_1+2x_2\\
\text{条件} \quad &-x_1+x_2 \geq 1,\\
& x_1 + x_2 \geq 3,\\
& 2x_1 + x_2 \leq 2,\\
& x_1, x_2 \geq 0.
\end{aligned}
$$

In [ ]:
A = np.array([[1, -1, 1, 0, 0],
            [-1, -1, 0, 1, 0],
            [2, 1, 0, 0, 1]], dtype=float)
b = np.array([-1, -3, 2], dtype=float)
c = np.array([3, 2, 0, 0, 0], dtype=float)

# 段階1
phase1 = AuxiliaryProblem(A, b)
B_idx, N_idx = phase1.find_feasible_basis()

# 段階2（通常の改訂単体法へ渡す）

solver = RevisedSimplex(A, b, c)
solver.B_idx,solver.N_idx = B_idx, N_idx

print("初期基底変数の添字:", [i+1 for i in B_idx])
print("初期非基底変数の添字:", [i+1 for i in N_idx])

x_opt, val_opt, iteration = solver.solve()

print("\n最適解:")
print(x_opt)

print("\n最適値:")
print(val_opt) #目的関数の符号を反転させているため、最適値も反転させる必要がある

### （補足）Scipyを用いた実装

In [1]:
import numpy as np
from scipy.optimize import linprog

$$
\begin{align*}
\text{最大化}\quad&z = x_1 + 2x_2 \\
\text{条件} \quad& x_1 + x_2 \leq 6 \\
\quad& x_1 + 3x_2 \leq 12 \\
\quad& 2x_1 + x_2 \leq 10\\
\quad& x_1, x_2 \geq 0
\end{align*}
$$

In [ ]:
#Scipyでは不等式制約のまま行列を定義
c = np.array([1, 2], dtype=float)
A = np.array([[1, 1], [1, 3], [2, 1]], dtype=float)
b = np.array([6, 12, 10], dtype=float)

In [ ]:
# SciPyでは最小化なので符号反転
result = linprog(
    c=-c,
    A_ub=A,
    b_ub=b,
    bounds=(0, None),
    method="revised simplex")#revised simplex法を指定（今は非推奨らしい）

print("status:")
print(result.message)

print("最適解:")
print(result.x)

print("最適値:")
print(-result.fun)